# 고속 스크리닝 실습

**High-throughput Screening · HTS · 다단 스크리닝**

값싼 판정으로 후보를 대량으로 걸러내고 남은 후보에 비싼 검증을 적용하는 단계적 탐색.

소재 분야에서 이해하기: 예측 모델로 1만 후보를 100개로 줄인 뒤 계산과 실험을 한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [Materials Project 용어집](https://docs.materialsproject.org/frequently-asked-questions/glossary-of-terms)

## 1. 값싼 판정으로 먼저 걸러내기

단계마다 비용과 정확도가 다른 필터를 쌓아 최종 후보를 남깁니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n = 20000
candidates = rng.random((n, 3))
def true_score(x):
    return 3 * x[:, 0] - 2 * x[:, 1] ** 2 + 1.5 * x[:, 0] * x[:, 2]

truth = true_score(candidates)
good = truth > np.percentile(truth, 99)      # 상위 1%가 진짜 좋은 후보
print('후보 %d개 중 진짜 좋은 후보 %d개' % (n, good.sum()))

COST = {'model': 0.001, 'calculation': 5.0, 'experiment': 400.0}   # 상대 비용(가상)
print('단계별 상대 비용:', COST)

In [ ]:
# 1단계: 값싼 예측 모델 (오차 큼)
model_estimate = truth + rng.normal(0, 1.2, n)
# 2단계: 계산 (오차 작음)
def calculation(indices):
    return truth[indices] + rng.normal(0, 0.25, len(indices))
# 3단계: 실험 (참값)
def experiment(indices):
    return truth[indices]

stage1 = np.argsort(-model_estimate)[:1000]
stage2 = stage1[np.argsort(-calculation(stage1))[:60]]
stage3 = stage2[np.argsort(-experiment(stage2))[:5]]

cost = n * COST['model'] + len(stage1) * COST['calculation'] + len(stage2) * COST['experiment']
print('단계별 통과 수: %d -> %d -> %d -> %d' % (n, len(stage1), len(stage2), len(stage3)))
print('총 비용 %.0f (전수 실험이라면 %.0f)' % (cost, n * COST['experiment']))
print('최종 5개 중 진짜 상위 1%%에 속한 개수 %d' % int(good[stage3].sum()))

## 2. 1단계를 너무 좁히면 좋은 후보를 잃습니다

In [ ]:
for keep in (100, 300, 1000, 3000):
    first = np.argsort(-model_estimate)[:keep]
    recall = good[first].sum() / good.sum()
    cost = n * COST['model'] + keep * COST['calculation'] + min(keep, 60) * COST['experiment']
    print('1단계 %4d개 통과 -> 진짜 좋은 후보 회수율 %.2f, 비용 %.0f' % (keep, recall, cost))
print('\n값싼 단계의 오차가 크면 통과 폭을 넓게 잡아야 좋은 후보를 잃지 않습니다.')
print('스크리닝 설계는 각 단계의 정확도와 비용을 함께 보고 결정해야 합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#high-throughput)을 여세요.